## Webpage Extraction and Embedding (PolyU SAO)

### 1. Extracting raw text data

In [2]:
from bs4 import BeautifulSoup
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client, PersistentClient

from concurrent.futures import ThreadPoolExecutor
import re

In [ ]:
sao_URL = "https://www.polyu.edu.hk/sao/"
start_idx, stop_idx = 36700, -900

docs = []
unwanted_metadata = ["language"]

def bs4_regex_enhance(html: str):
    soup = BeautifulSoup(html, "lxml")                  # Strip the html syntax
    text = re.sub(r"\n\n+", "\n\n", soup.text).strip()  # Strip the excess newlines
    return text

sao_loader = RecursiveUrlLoader(
    url=sao_URL,
    base_url=sao_URL,
    prevent_outside=True,
    exclude_dirs=[
        sao_URL+"news-and-events",
        sao_URL+"News-and-Events", 
        sao_URL+"Sitemap",
        sao_URL+"sitemap",
        sao_URL+"Search-Result",
        sao_URL+"Personal-Information-Collection-Statement",
        sao_URL+"Counselling-and-Wellness-Section/PolyU-Asian-Universities-Water-Polo-Invitational-Tournament",
        sao_URL+"Student-Resources-and-Support-Section/Outstanding-Student-Academy",
        sao_URL+"Non-local-Student-Services/Event-Highlight",
    ],
    extractor=bs4_regex_enhance
)

docs_lazy = sao_loader.lazy_load()
for doc in docs_lazy:
    print(doc.metadata.get('source'))
    
    doc.page_content = doc.page_content[start_idx:stop_idx]
    for key in unwanted_metadata:
        del doc.metadata[key]
    docs.append(doc)

In [16]:
print(f"Extracted number of webpages in SAO: {len(docs)}")
print(docs[65].page_content[:])
print(docs[65].metadata.get('source'))

'''
idx = 5
print(docs[idx].metadata.get('source'))
print(docs[idx].page_content)
#print(docs[idx].page_content[1300:-300])
'''

Extracted number of webpages in SAO: 265


Quick Access

Start main content

													Home
												

													Careers and Placement
												

													Plan for Your Careers
												

Plan for Your Careers

 

The Careers and Placement Section (CPS), Student Affairs Office (SAO) aims to empower students to discover and embark upon impactful careers through the provision of experiential learning, mentoring and networking opportunities in collaboration with the broader PolyU community.

 

"INSPIRE" Mentorship Programme

The “INSPIRE” Mentorship Programme aims to enhance students’ holistic education experience and facilitate their personal, academic, and professional development through a robust network of role models and prominent leaders of the PolyU community.

                                                                        Let's INSPIRE!
                                                                    

Career Support for Students with SEN

CPS, SAO 

"\nidx = 5\nprint(docs[idx].metadata.get('source'))\nprint(docs[idx].page_content)\n#print(docs[idx].page_content[1300:-300])\n"

### 2. Text Splitting

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_documents(docs)

for i, chunk in enumerate(chunks):
    source = chunk.metadata.get("source", "N/A")
    url_end = re.search(r'/([^/]+)/?$', source).group(1)
    chunk.metadata["chunk_id"] = f"PolyU_SAO_{url_end}_chunk_{i}"

    chunk.page_content = f"--- PolyU SAO Website URL: {source} ---\n\n{chunk.page_content}"

In [18]:
print(chunks[10])

page_content='EAGLE Global Youth Leadership Hub
												

													Cross-institutional Student-led Social Projects
												

Cross-institutional Student-led Social Projects

 

Cross-institutional Student-led Social Projects 
The call for applications for Cross-institutional Student-led Social Projects (CSSP) 2025/26 offers a maximum of HK$80,000 to award subsidies for selected projects to be undertaken by project groups from UGC-funded universities in Hong Kong.
The purpose of CSSP is to foster mutual understanding and collaboration among students from universities, and to encourage creative and constructive initiatives that enhance learning and holistic development while making a positive social impact.
Organised by:
EAGLE Global Youth Leadership Hub (the EAGLE Hub),
Student Development Section (SDS), Student Affairs Office (SAO),
The Hong Kong Polytechnic University (PolyU)
Supported by:
University & College YMCA Department (Uni-Y), 
The Chinese YMCA of Hong Kong' metada

### 3. Document Embedding in Chroma

In [13]:
SINGLE = True # Change to True if you want to use single chroma database for all documents
collection_name = "sao_documents" if not SINGLE else "vaa_documents"

In [14]:
embedding_function = OllamaEmbeddings(model="bge-m3:567m") # Please OPEN Ollama first!!

client = Client(Settings())
client = PersistentClient(path="../chroma_db")
collection = client.get_collection(name=collection_name)

client.get_collection(name=collection_name).count()

4291

In [ ]:
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

for i, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content], 
        metadatas=[chunk.metadata], 
        embeddings=[embeddings[i]],
        ids=[str(i+ 500)]
    )

print(f"Added {len(chunks)} chunks into ChromaDB to {collection_name}")

### 4. Simple Testing

In [15]:
vectorStore = Chroma(
    collection_name=collection_name, 
    client=client, 
    embedding_function=embedding_function)

query = "How can I apply for residential hall in PolyU?"
results = vectorStore.similarity_search(query, k=5)

for result in results:
    print("========================================================")
    print(f"Content: {result.page_content}...")
    print(f"Source: {result.metadata.get('source')}")
    print(f"Chunk ID: {result.metadata.get('chunk_id')}\n")

Content: --- PolyU SAO Website URL: https://www.polyu.edu.hk/sao/student-resources-and-support-section/residential-life/hall-admission/hall-applications/ ---

t Resources and Support
												

													Residential Life
												

													Hall Admission
												

													Hall Applications
												

Hall Applications

Students are strongly encouraged to have hall-life experience during their university years, which is both memorable and rewarding. The Student Halls of Residence do not just provide students with an accommodation, but a vibrant community with abundant opportunities for them to grow and learn.

 
Application for Hall Residence 2025/26 – Full-time undergraduate students

Student Type

Application Period

Current students
(including Special Readmission Scheme (SRS) &
Readmission Scheme of CURI Residential College (RSCRC))
2 May (10:00am) to 19 May 2025 (11:59pm)

Non-local new students

11 Jun (10:00am) - 25 Jul 2025 (11:59pm) - Phase 1
          